In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import scanpy as sc
import anndata

import mudata

from scenicplus.RSS import (regulon_specificity_scores, plot_rss)
from scenicplus.plotting.dotplot import heatmap_dotplot

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))

In [ ]:
indir = f'{projdir}/gene_networks/scenic/Snakemake/'
scplus_mdata = mudata.read(f'{indir}/scplusmdata.h5mu')

In [ ]:
scplus_mdata

In [ ]:
d_regulon = scplus_mdata.uns["direct_e_regulon_metadata"]
d_regulon.shape

In [ ]:
d_regulon.head()

In [ ]:
e_regulon = scplus_mdata.uns["extended_e_regulon_metadata"]
e_regulon.shape

In [ ]:
e_regulon.head()

In [ ]:
clusters_path = f'{projdir}/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
clusters['coarse_v7'].value_counts()

In [ ]:
clusters['coarse_donor'] = [f'{x}_{y}' for x,y in zip (clusters['coarse_v7'], clusters['assignment'])]

In [ ]:
scplus_mdata.obs['coarse'] = clusters.reindex(scplus_mdata.obs.index)['coarse_v7']
scplus_mdata.obs['cluster'] = clusters.reindex(scplus_mdata.obs.index)['clusters_v7']
scplus_mdata.obs['coarse_donor'] = clusters.reindex(scplus_mdata.obs.index)['coarse_donor']

In [ ]:
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed''Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
scplus_mdata.obs['coarse_donor_ordered'] = pd.Categorical(
    scplus_mdata.obs['coarse_donor'],
    categories=clusts,
    ordered=True
)

In [ ]:
eRegulon_gene_AUC = anndata.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC

In [ ]:
sc.pp.neighbors(eRegulon_gene_AUC, use_rep = "X")
sc.tl.umap(eRegulon_gene_AUC)

In [ ]:
sc.pl.umap(eRegulon_gene_AUC, color = 'coarse')

In [ ]:
sc.pl.umap(eRegulon_gene_AUC, color = 'cluster')

In [ ]:
rss = regulon_specificity_scores(
    scplus_mudata = scplus_mdata,
    variable = 'coarse',
    modalities = ["direct_gene_based_AUC", "extended_gene_based_AUC"]
)

In [ ]:
rss.shape

In [ ]:
rss.head()

In [ ]:
plot_rss(
    data_matrix = rss,
    top_n = 3,
    num_columns = 5
)

In [ ]:
sc.pl.umap(eRegulon_gene_AUC,
           color = list(set([x for xs in [rss.loc[ct].sort_values()[0:2].index for ct in rss.index] for x in xs ])))

In [ ]:
rss = regulon_specificity_scores(
    scplus_mudata = scplus_mdata,
    variable = 'cluster',
    modalities = ["direct_gene_based_AUC", "extended_gene_based_AUC"]
)
plot_rss(
    data_matrix = rss,
    top_n = 3,
    num_columns = 5
)

In [ ]:
sc.pl.umap(eRegulon_gene_AUC,
           color = list(set([x for xs in [rss.loc[ct].sort_values()[0:2].index for ct in rss.index] for x in xs ])))

In [ ]:
heatmap_dotplot(
    scplus_mudata = scplus_mdata,
    color_modality = "direct_gene_based_AUC",
    size_modality = "direct_region_based_AUC",
    group_variable = 'coarse',
    eRegulon_metadata_key = "direct_e_regulon_metadata",
    color_feature_key = "Gene_signature_name",
    size_feature_key = "Region_signature_name",
    feature_name_key = "eRegulon_name",
    sort_data_by = "direct_gene_based_AUC",
    orientation = "horizontal",
    figsize = (40, 10)
)

In [ ]:
heatmap_dotplot(
    scplus_mudata = scplus_mdata,
    color_modality = "direct_gene_based_AUC",
    size_modality = "direct_region_based_AUC",
    group_variable = 'cluster',
    eRegulon_metadata_key = "direct_e_regulon_metadata",
    color_feature_key = "Gene_signature_name",
    size_feature_key = "Region_signature_name",
    feature_name_key = "eRegulon_name",
    sort_data_by = "direct_gene_based_AUC",
    orientation = "horizontal",
    figsize = (30, 10)
)

In [ ]:
heatmap_dotplot(
    scplus_mudata = scplus_mdata,
    color_modality = "direct_gene_based_AUC",
    size_modality = "direct_region_based_AUC",
    group_variable = 'coarse_donor_ordered',
    eRegulon_metadata_key = "direct_e_regulon_metadata",
    color_feature_key = "Gene_signature_name",
    size_feature_key = "Region_signature_name",
    feature_name_key = "eRegulon_name",
    sort_data_by = "direct_gene_based_AUC",
    orientation = "horizontal",
    figsize = (30, 12)
)

In [ ]:
from scenicplus.scenicplus_class import mudata_to_scenicplus

In [ ]:
indir = f'{projdir}/gene_networks/scenic/Snakemake'
scplus_obj = mudata_to_scenicplus(
    mdata = scplus_mdata,
    path_to_cistarget_h5 = f'{indir}/ctx_results.hdf5',
    path_to_dem_h5 = f'{indir}/dem_results.hdf5'
)

In [ ]:
scplus_obj